## Model Based Handling:  Small explanation is in the file 'all_about_it.md' and mostly explained here with code executions

Instead of preprocessing the data to replace `NaN`s with **SimpleImputer**, **KNNImputer**, or **IterativeImputer**, you feed the dataset with missing values directly into the machine learning algorithm, and the model handles them internally during training and prediction.

*You don't even do all the boring imputation process, just plug data with missing values to a ML model, it does all you work.*

### How do Algorithms Handle `NaN`s internally?
Modern tree-based models like **XGBoost**, **LightGBM**, **CatBoost**, and modern **Histogram-based Gradient Boosting** (HistGradientBoostingClassifier / Regressor in Scikit-Learn)—have built-in mechanisms to treat missingness as a signal.

#### 1. Default Direction Routing (XGBoost / LightGBM)
When building decision tree:
1) At every split node (e.g. Is Experience > 5?), the algorithm evaluates with direction (left branch vs Right branch) gives a lower value. 
2) It tests sending all missing values to left branch, then test sending all them to right branch.
3) Whichever branch yields the best overall performance becomes the default path for any row where the feature is missing.

#### 2. Missing as separate Category/Value
Tree algorithms can treat `NaN`as a distinct value rather than forcing it to match an existing number:
- For categorical features, `NaM` is treated as its own separate category (e.g. Missing).
- For continues numerical features, the model can assign missing values to dedicated histogram bins.

#### Step by step example:

| **Customer** | **Income** | **Credit Score**              | **Defaulted?** |
| ------------ | ---------- | ----------------------------- | -------------- |
| **C1**       | $50,000    | 720                           | No             |
| **C2**       | $30,000    | 580                           | Yes            |
| **C3**       | $45,000    | **`NaN`** (No credit history) | Yes            |

- Imputation Approach: Might fill Credit Score with the average ($650$), treating Customer 3 as having decent credit.
- Model-Based Handling Approach: The model notices that customers without a credit score frequently default. It learns a rule:
    - If Credit Score is missing -? Route directly to High-Risk Leaf

It preserves the information that the value was missing for a reason (often called Missingness Indicator behavior).

In [2]:
# Example dataset

import numpy as np
import pandas as pd

# Create a sample dataset with correlated features and missing values
data = {
    'Experience_Years': [1, 2, 3, 10, 11, 12, 20, 22],
    'Age': [22, 24, 25, 33, 35, 36, 48, 52],
    'Salary': [30000, 35000, 38000, 85000, np.nan, 95000, 150000, 160000] # Missing senior-level salary
}

df = pd.DataFrame(data)

print("=== ORIGINAL DATASET WITH MISSING VALUES ===")
print(df)

=== ORIGINAL DATASET WITH MISSING VALUES ===
   Experience_Years  Age    Salary
0                 1   22   30000.0
1                 2   24   35000.0
2                 3   25   38000.0
3                10   33   85000.0
4                11   35       NaN
5                12   36   95000.0
6                20   48  150000.0
7                22   52  160000.0


In [5]:
from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb

# Features (X) and Target (y)
X = df[['Age', 'Salary']] # Salary contains a NaN
y = df['Experience_Years']

# -------------------------------------------------------------
# Method A: Scikit-Learn's HistGradientBoosting (Native NaN support)
# -------------------------------------------------------------
hgb_model = HistGradientBoostingRegressor(random_state=42)
# Fits directly with NaNs present — NO SimpleImputer needed!
hgb_model.fit(X, y)
print("HistGradientBoosting Predictions:", hgb_model.predict(X))

hgb_model_df=pd.DataFrame({
    'Age': X['Age'],
    'Salary': X['Salary'],
    'Predicted_Experience_Years': hgb_model.predict(X)
})
print("\n=== HistGradientBoosting Predictions ===")
print(hgb_model_df)


HistGradientBoosting Predictions: [10.125 10.125 10.125 10.125 10.125 10.125 10.125 10.125]

=== HistGradientBoosting Predictions ===
   Age    Salary  Predicted_Experience_Years
0   22   30000.0                      10.125
1   24   35000.0                      10.125
2   25   38000.0                      10.125
3   33   85000.0                      10.125
4   35       NaN                      10.125
5   36   95000.0                      10.125
6   48  150000.0                      10.125
7   52  160000.0                      10.125


In [7]:
# -------------------------------------------------------------
# Method B: XGBoost (Native NaN support)
# -------------------------------------------------------------
xgb_model = xgb.XGBRegressor(random_state=42)
# XGBoost automatically learns the best direction for missing values
xgb_model.fit(X, y)
print("XGBoost Predictions:", xgb_model.predict(X))

xgb_model_df=pd.DataFrame({
    'Age': X['Age'],
    'Salary': X['Salary'],
    'Predicted_Experience_Years': xgb_model.predict(X)
})  

print("\n=== XGBoost Predictions ===")
print(xgb_model_df)

XGBoost Predictions: [ 1.0011051  2.000208   2.9999938 10.000054  11.000047  11.999836
 19.999971  21.998783 ]

=== XGBoost Predictions ===
   Age    Salary  Predicted_Experience_Years
0   22   30000.0                    1.001105
1   24   35000.0                    2.000208
2   25   38000.0                    2.999994
3   33   85000.0                   10.000054
4   35       NaN                   11.000047
5   36   95000.0                   11.999836
6   48  150000.0                   19.999971
7   52  160000.0                   21.998783
